# Understanding Supervised Fine-Tuning (SFT)

This notebook demonstrates how supervised fine-tuning works for language models.

## What is SFT?

Supervised Fine-Tuning (SFT) is the process of adapting a pre-trained language model to a specific task by training it on instruction-following examples. The model learns to:
- Follow instructions
- Generate appropriate responses
- Adapt to specific domains or styles

## Key Concepts

1. **Next-token prediction**: Model predicts the next token given previous context
2. **Prompt masking**: We only compute loss on response tokens, not the prompt
3. **LoRA**: Efficient fine-tuning by training low-rank adapters

Let's see it in action!

In [ ]:
# Setup
import sys
sys.path.append('..')

import torch
from transformers import TrainingArguments
from datasets import Dataset

from src.models.language import LanguageModel
from src.data.processors.text import TextProcessor
from src.core.sft.trainer import SFTTrainer
from src.core.sft.collator import DataCollatorForSFT, create_sft_dataset
from src.evaluation.metrics.text import TextMetrics

## Step 1: Load a Small Model

We'll use GPT-2 (124M parameters) with LoRA for efficient fine-tuning.

In [ ]:
# Load model with LoRA
model = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=True,
    lora_config={
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
    }
)

print(f"Total parameters: {model.num_parameters:,}")
print(f"Trainable parameters: {model.num_trainable_parameters:,}")
print(f"Trainable %: {100 * model.num_trainable_parameters / model.num_parameters:.2f}%")

## Step 2: Create a Training Dataset

We'll create a small dataset of instruction-response pairs.

In [ ]:
# Create training data
train_examples = [
    {"prompt": "What is the capital of France?", "response": "The capital of France is Paris."},
    {"prompt": "What is 2 + 2?", "response": "2 + 2 equals 4."},
    {"prompt": "Who wrote Romeo and Juliet?", "response": "William Shakespeare wrote Romeo and Juliet."},
    {"prompt": "What is the largest planet?", "response": "Jupiter is the largest planet in our solar system."},
    {"prompt": "What is Python?", "response": "Python is a high-level programming language known for its simplicity."},
]

eval_examples = [
    {"prompt": "What is the capital of Spain?", "response": "The capital of Spain is Madrid."},
    {"prompt": "What is 5 + 3?", "response": "5 + 3 equals 8."},
]

print(f"Training examples: {len(train_examples)}")
print(f"Eval examples: {len(eval_examples)}")

## Step 3: Tokenize the Data

Convert text to token IDs and create labels with prompt masking.

In [ ]:
# Tokenize datasets
train_tokenized = create_sft_dataset(
    examples=train_examples,
    tokenizer=model.tokenizer,
    max_length=128,
)

eval_tokenized = create_sft_dataset(
    examples=eval_examples,
    tokenizer=model.tokenizer,
    max_length=128,
)

# Convert to Dataset
train_dataset = Dataset.from_list(train_tokenized)
eval_dataset = Dataset.from_list(eval_tokenized)

print("Example tokenized data:")
example = train_tokenized[0]
print(f"Input IDs length: {len(example['input_ids'])}")
print(f"Labels (first 20): {example['labels'][:20]}")
print("Note: -100 means 'ignore in loss computation' (prompt tokens)")

## Step 4: Setup Training

Create the data collator and training arguments.

In [ ]:
# Data collator
data_collator = DataCollatorForSFT(
    tokenizer=model.tokenizer,
    max_length=128,
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./outputs/sft_tutorial",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    warmup_steps=10,
    logging_steps=1,
    eval_steps=5,
    evaluation_strategy="steps",
    save_strategy="no",  # Don't save checkpoints for this demo
)

print("Training configuration:")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

## Step 5: Create Trainer and Train

Use our custom SFT trainer with detailed logging.

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=model.model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=model.tokenizer,
    data_collator=data_collator,
    loss_type="causal_lm",
    log_predictions=False,  # Disable for this demo
)

print("Starting training...")
trainer.train()
print("Training complete!")

## Step 6: Evaluate the Model

Let's see how the model performs after training.

In [ ]:
# Evaluate
metrics = trainer.evaluate()

print("\nEvaluation Metrics:")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

## Step 7: Generate Responses

Test the fine-tuned model with some prompts.

In [ ]:
# Test generation
model.eval()

test_prompts = [
    "What is the capital of Italy?",
    "What is 10 + 5?",
    "Who invented the telephone?",
]

processor = TextProcessor(model.tokenizer, max_length=128)

print("\nGenerated Responses:")
print("=" * 60)

for prompt in test_prompts:
    # Encode
    encoded = processor.tokenize(prompt, return_tensors="pt")
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            max_new_tokens=30,
            temperature=0.7,
            do_sample=True,
        )
    
    # Decode
    generated_text = processor.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated_text}")
    print("-" * 60)

## Step 8: Analyze Training Metrics

Visualize how the model improved during training.

In [ ]:
import matplotlib.pyplot as plt

# Get training metrics
metrics = trainer.get_training_metrics()

if metrics['steps']:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Loss
    if metrics['losses']:
        axes[0, 0].plot(metrics['steps'], metrics['losses'])
        axes[0, 0].set_title('Training Loss')
        axes[0, 0].set_xlabel('Step')
        axes[0, 0].set_ylabel('Loss')
    
    # Accuracy
    if metrics['accuracies']:
        axes[0, 1].plot(metrics['steps'], metrics['accuracies'])
        axes[0, 1].set_title('Token Accuracy')
        axes[0, 1].set_xlabel('Step')
        axes[0, 1].set_ylabel('Accuracy')
    
    # Perplexity
    if metrics['perplexities']:
        axes[1, 0].plot(metrics['steps'], metrics['perplexities'])
        axes[1, 0].set_title('Perplexity')
        axes[1, 0].set_xlabel('Step')
        axes[1, 0].set_ylabel('Perplexity')
    
    # Gradient Norm
    if metrics['grad_norms']:
        axes[1, 1].plot(metrics['steps'], metrics['grad_norms'])
        axes[1, 1].set_title('Gradient Norm')
        axes[1, 1].set_xlabel('Step')
        axes[1, 1].set_ylabel('Grad Norm')
    
    plt.tight_layout()
    plt.show()
else:
    print("No metrics to plot")

## Summary

In this notebook, we:
1. ✅ Loaded GPT-2 with LoRA adapters (only 0.5% of parameters trainable)
2. ✅ Created a small instruction-following dataset
3. ✅ Tokenized data with proper prompt masking
4. ✅ Trained the model using SFT
5. ✅ Evaluated performance
6. ✅ Generated responses from the fine-tuned model
7. ✅ Analyzed training dynamics

## Key Takeaways

- **SFT is straightforward**: Just next-token prediction on instruction data
- **Prompt masking is crucial**: Only compute loss on responses
- **LoRA is efficient**: Train only 0.5% of parameters
- **Small datasets work**: Even 5 examples show improvement

## Next Steps

- Try with larger datasets (Alpaca, Dolly, etc.)
- Experiment with different LoRA ranks
- Try different prompt templates
- Move to reward modeling and RLHF!

See `02_reward_modeling.ipynb` for the next technique.